In [461]:
# Imports and pip installations (if needed)
import numpy as np
import pandas as pd
from sklearn import model_selection, linear_model, svm, neural_network, neighbors
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

# Part 1: Load the dataset

In [462]:
# Load the dataset (load remotely or locally)
data = pd.read_csv("iris.csv")
# Output the first 15 rows of the data
display(data.head(15))
# Display a summary of the table information (number of datapoints, etc.)
display(data.describe())


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa
5,5.4,3.9,1.7,0.4,setosa
6,4.6,3.4,1.4,0.3,setosa
7,5.0,3.4,1.5,0.2,setosa
8,4.4,2.9,1.4,0.2,setosa
9,4.9,3.1,1.5,0.1,setosa


,sepal_length,sepal_width,petal_length,petal_width
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.054000,3.758667,1.198667
std,0.828066,0.433594,1.764420,0.763161
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


## About the dataset
Explain what the data is in your own words. What are your features and labels? What is the mapping of your labels to the actual classes?

**Explanation**:  
The data is various measurements of characteristics of flowers and the species of that flower. The features are these characteristic measurements, namely sepal length/width and petal length/width. The labels are the species column, whose classes are 'setosa', 'versicolor', and 'virginica'.

# Part 2: Split the dataset into train and test

In [463]:
# Take the dataset and split it into our features (X) and label (y)
X = data.drop('species', axis=1)
y = data['species']
# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=0.2, train_size=0.8, random_state=383)
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape: {y_test.shape}')

KFold_shuffle = model_selection.KFold(shuffle=True) # used later for cross validation 

X_train shape: (120, 4)
X_test shape: (30, 4)
y_train shape: (120,)
y_test shape: (30,)


# Part 3: Logistic Regression

In [464]:
# i. Use sklearn to train a LogisticRegression model on the training set
log_reg = linear_model.LogisticRegression(max_iter=150).fit(X_train, y_train)
# ii. For a sample datapoint, predict the probabilities for each possible class
datapoint = [5.5, 2.9, 4.5, 1.1]
print(f'sample datapoint: {datapoint}')
print(f'prediction based on sample datapoint: {log_reg.predict([datapoint])}')
# iii. Report on the score for Logistic regression model, what does the score measure?
print(f'\nmodel score: {log_reg.score(X_test, y_test)}')
cross_val_scores = np.array([model_selection.cross_val_score(log_reg, X_train, y_train, cv=KFold_shuffle) for i in range(2)])
print(f'model cross val scores: \n{cross_val_scores}')
print(f'cross val mean: {np.average(cross_val_scores)}')
# iv. Extract the coefficents and intercepts for the boundary line(s)
print(f'\nmodel features: {log_reg.feature_names_in_}')
print(f'model classes: {log_reg.classes_}')
print(f'model coefficients: \n{log_reg.coef_}')
print(f'model intercept: {log_reg.intercept_}')

sample datapoint: [5.5, 2.9, 4.5, 1.1]
prediction based on sample datapoint: ['versicolor']

model score: 1.0
model cross val scores: 
[[0.95833333 0.91666667 0.95833333 1.         0.875     ]
 [0.95833333 1.         0.83333333 1.         0.95833333]]
cross val mean: 0.9458333333333334

model features: ['sepal_length' 'sepal_width' 'petal_length' 'petal_width']
model classes: ['setosa' 'versicolor' 'virginica']
model coefficients: 
[[-0.44793175  0.81556739 -2.29417533 -0.99904282]
 [ 0.54757157 -0.43796954 -0.13663657 -0.87760677]
 [-0.09963982 -0.37759786  2.43081189  1.8766496 ]]
model intercept: [  9.57671112   1.89496364 -11.47167476]


**Report**:  
We get a model score of 1.0 on the test set, which indicates that the model fits the data perfectly. However, this is likely due to the very small size of the test set. Upon carrying out cross validation, we can see that the average model score across shuffles of the data is in the ~.95, which indicates a relatively good, but not perfect fit with our logistic regression model.

*Equation(s) of boundary line(s)*:  
$$l_s = \text{sepal length}, w_s = \text{sepal width}, l_p = \text{petal length}, w_p = \text{petal width}$$
$$ b_{\text{setosa}} = \frac{1}{1+e^{-(9.5767 + (-0.4479)l_s + 0.8156w_s + (-2.2942)l_p + (-0.9990)w_p)}} $$  
$$ b_{\text{versicolor}} = \frac{1}{1+e^{-(1.8950 + 0.5476l_s + (-0.4380)w_s + (-0.1366)l_p + (-0.8776)w_p)}} $$  
$$ b_{\text{virginica}} = \frac{1}{1+e^{-((-11.4717) + (-0.0996)l_s + (-0.3776)w_s + 2.4308l_p + 1.8766w_p)}} $$

# Part 4: Support Vector Machine

In [465]:
# i. Use sklearn to train a Support Vector Classifier on the training set
svc = svm.SVC().fit(X_train, y_train)
# ii. For a sample datapoint, predict the probabilities for each possible class
print(f'sample datapoint: {datapoint}')
print(f'prediction based on sample datapoint: {svc.predict([datapoint])}')
# iii. Report on the score for the SVM, what does the score measure?
print(f'\nmodel score: {svc.score(X_test, y_test)}')
cross_val_scores = np.array([model_selection.cross_val_score(svc, X_train, y_train, cv=KFold_shuffle) for i in range(2)])
print(f'model cross val scores: \n{cross_val_scores}')
print(f'cross val mean: {np.average(cross_val_scores)}')

sample datapoint: [5.5, 2.9, 4.5, 1.1]
prediction based on sample datapoint: ['versicolor']

model score: 1.0
model cross val scores: 
[[0.95833333 0.95833333 0.95833333 0.91666667 0.91666667]
 [0.875      0.95833333 1.         0.91666667 0.91666667]]
cross val mean: 0.9375


**Report**:  
Again we get a model score of 1.0 using the test sets, which again is likely due to the incredibly small dataset size. Upon examining cross validation scores, we see an average model score across shuffles of ~.94, which again indicates a generally good fit for this model, but not perfect.

# Part 5: Neural Network

In [466]:
# i. Use sklearn to train a Neural Network (MLP Classifier) on the training set
mlpc = neural_network.MLPClassifier(max_iter=750).fit(X_train, y_train)
# ii. For a sample datapoint, predict the probabilities for each possible class
print(f'sample datapoint: {datapoint}')
print(f'prediction based on sample datapoint: {mlpc.predict([datapoint])}')
# iii. Report on the score for the Neural Network, what does the score measure?
print(f'\nmodel score: {mlpc.score(X_test, y_test)}')
cross_val_scores = np.array([model_selection.cross_val_score(mlpc, X_train, y_train, cv=KFold_shuffle) for i in range(2)])
print(f'model cross val scores: \n{cross_val_scores}')
print(f'cross val mean: {np.average(cross_val_scores)}')
# iv: Experiment with different options for the neural network, report on your best configuration
mlpc2 = neural_network.MLPClassifier(max_iter=750, learning_rate='adaptive', activation='logistic').fit(X_train, y_train)
print(f'\nmodel 2 score: {mlpc2.score(X_test, y_test)}')
cross_val_scores = np.array([model_selection.cross_val_score(mlpc2, X_train, y_train, cv=KFold_shuffle) for i in range(2)])
print(f'model 2 cross val scores: \n{cross_val_scores}')
print(f'cross val mean: {np.average(cross_val_scores)}')

sample datapoint: [5.5, 2.9, 4.5, 1.1]
prediction based on sample datapoint: ['versicolor']

model score: 1.0
model cross val scores: 
[[1.         0.95833333 1.         0.83333333 1.        ]
 [1.         0.91666667 1.         1.         0.91666667]]
cross val mean: 0.9625

model 2 score: 1.0
model 2 cross val scores: 
[[0.95833333 1.         0.95833333 1.         0.91666667]
 [1.         0.95833333 1.         0.91666667 1.        ]]
cross val mean: 0.9708333333333334


**Report**:  
Once again, we get a model score using the test set of 1.0, mostly due to the dataset size. Upon performing cross validation, we find an average score of ~.96, once again indicating a generally good (but not perfect) fit for the data using the neural network.

As for tinkering with the model parameters to get a better fit, I wasn't sure what to tweak. However, I eventually found this configuration: (learning_rate = 'adaptive', activation = 'logistic') led to a slight increase in average cross validation score (~.97 instead of .96). Despite this result, I am not exactly sure why this is the case.

# Part 6: K-Nearest Neighbors

In [ ]:
# i. Use sklearn to 'train' a k-Neighbors Classifier
# Note: KNN is a nonparametric model and technically doesn't require training
# fit will essentially load the data into the model see link below for more information
# https://stats.stackexchange.com/questions/349842/why-do-we-need-to-fit-a-k-nearest-neighbors-classifier
knc = neighbors.KNeighborsClassifier().fit(X_train, y_train)
# ii. For a sample datapoint, predict the probabilities for each possible class
print(f'sample datapoint: {datapoint}')
print(f'prediction based on sample datapoint: {knc.predict([datapoint])}')
# iii. Report on the score for kNN, what does the score measure?
print(f'\nmodel score: {knc.score(X_test, y_test)}')
cross_val_scores = np.array([model_selection.cross_val_score(knc, X_train, y_train, cv=KFold_shuffle) for i in range(2)])
print(f'model cross val scores: \n{cross_val_scores}')
print(f'cross val mean: {np.average(cross_val_scores)}')

sample datapoint: [5.5, 2.9, 4.5, 1.1]
prediction based on sample datapoint: ['versicolor']

model score: 1.0
model cross val scores: 
[[0.91666667 0.95833333 0.91666667 0.95833333 0.95833333]
 [1.         0.91666667 1.         0.95833333 0.95833333]]
cross val mean: 0.9541666666666668


**Report**:  
Finally, again we get a model score of 1.0 due to small dataset size. Upon cross validation scoring, we find an average score of ~.95, once again indicating a generally good (but not perfect) fit for this particular model.

# Part 7: Conclusions and takeaways

In your own words describe the results of the notebook. Which model(s) performed the best on the dataset? Why do you think that is? Did anything surprise you about the exercise?

**Answer**:  
Generally, all of the models did fairly well in modeling this dataset, likely due to its small size. However, it seems that the second configuration I found for the neural network results in the best overall average cross validation score, meaning it models the data better than the rest. I would assume this is because neural networks can essentially model any continuous function (or boundary line). Thus, by training a neural network on this data, we in turn create a model that simulates the best boundary line(s) to fit each class of flower.